In [0]:
# ---------------------------------------------------------------------------
# 1. Configuration & Landing Zone Folder Preparation
# ---------------------------------------------------------------------------
import csv
import json
import os
import random
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

CATALOG_NAME = "aircargo_dev"
BRONZE_TELEMETRY_TABLE = f"{CATALOG_NAME}.bronze.flight_telemetry_raw"

LANDING_BASE_PATH = "/Volumes/aircargo_dev/landing/raw_files"
AIRPORTS_DIR = f"{LANDING_BASE_PATH}/airports"
WEATHER_DIR = f"{LANDING_BASE_PATH}/weather"
MANIFESTS_DIR = f"{LANDING_BASE_PATH}/cargo_manifests"

# Create separate landing folders per dataset (required for Auto Loader)
for folder in [AIRPORTS_DIR, WEATHER_DIR, MANIFESTS_DIR]:
    os.makedirs(folder, exist_ok=True)

batch_ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
print(f"Landing directories ready. Batch timestamp: {batch_ts}")

In [0]:
# ---------------------------------------------------------------------------
# 2. Generate Reference Master Data: Airports (CSV)
# ---------------------------------------------------------------------------
airports_data = [
    {"airport_iata": "WAW", "airport_icao": "EPWA", "airport_name": "Warsaw Chopin Airport", "city": "Warsaw", "country": "Poland", "latitude": 52.1657, "longitude": 20.9671, "cargo_hub_tier": "REGIONAL"},
    {"airport_iata": "FRA", "airport_icao": "EDDF", "airport_name": "Frankfurt Airport", "city": "Frankfurt", "country": "Germany", "latitude": 50.0379, "longitude": 8.5622, "cargo_hub_tier": "MEGA_HUB"},
    {"airport_iata": "LEJ", "airport_icao": "EDDP", "airport_name": "Leipzig/Halle Airport", "city": "Leipzig", "country": "Germany", "latitude": 51.4324, "longitude": 12.2416, "cargo_hub_tier": "MEGA_HUB"},
    {"airport_iata": "AMS", "airport_icao": "EHAM", "airport_name": "Amsterdam Airport Schiphol", "city": "Amsterdam", "country": "Netherlands", "latitude": 52.3105, "longitude": 4.7683, "cargo_hub_tier": "MEGA_HUB"},
    {"airport_iata": "LHR", "airport_icao": "EGLL", "airport_name": "London Heathrow Airport", "city": "London", "country": "United Kingdom", "latitude": 51.4700, "longitude": -0.4543, "cargo_hub_tier": "MAJOR"},
    {"airport_iata": "CDG", "airport_icao": "LFPG", "airport_name": "Paris Charles de Gaulle", "city": "Paris", "country": "France", "latitude": 49.0097, "longitude": 2.5479, "cargo_hub_tier": "MEGA_HUB"},
    {"airport_iata": "ORD", "airport_icao": "KORD", "airport_name": "Chicago O'Hare International", "city": "Chicago", "country": "United States", "latitude": 41.9742, "longitude": -87.9073, "cargo_hub_tier": "MEGA_HUB"},
    {"airport_iata": "CVG", "airport_icao": "KCVG", "airport_name": "Cincinnati/Northern Kentucky", "city": "Hebron", "country": "United States", "latitude": 39.0488, "longitude": -84.6678, "cargo_hub_tier": "MEGA_HUB"},
    {"airport_iata": "MEM", "airport_icao": "KMEM", "airport_name": "Memphis International Airport", "city": "Memphis", "country": "United States", "latitude": 35.0424, "longitude": -89.9767, "cargo_hub_tier": "MEGA_HUB"},
    {"airport_iata": "HKG", "airport_icao": "VHHH", "airport_name": "Hong Kong International", "city": "Hong Kong", "country": "Hong Kong", "latitude": 22.3080, "longitude": 113.9185, "cargo_hub_tier": "MEGA_HUB"},
    {"airport_iata": "DXB", "airport_icao": "OMDB", "airport_name": "Dubai International Airport", "city": "Dubai", "country": "UAE", "latitude": 25.2532, "longitude": 55.3657, "cargo_hub_tier": "MEGA_HUB"},
    {"airport_iata": "SIN", "airport_icao": "WSSS", "airport_name": "Singapore Changi Airport", "city": "Singapore", "country": "Singapore", "latitude": 1.3644, "longitude": 103.9915, "cargo_hub_tier": "MEGA_HUB"},
]

airports_file = f"{AIRPORTS_DIR}/airports_{batch_ts}.csv"
fieldnames = list(airports_data[0].keys())

with open(airports_file, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(airports_data)

print(f"Wrote {len(airports_data)} airports to {airports_file}")

In [0]:
# ---------------------------------------------------------------------------
# 3. Generate Airport Weather Snapshots (JSON Lines / NDJSON)
# ---------------------------------------------------------------------------
conditions = ["CLEAR", "CLOUDY", "LIGHT_RAIN", "FOG", "HEAVY_SNOW", "THUNDERSTORM"]
observed_at = datetime.now(timezone.utc).isoformat()
weather_records = []

for ap in airports_data:
    condition = random.choice(conditions)
    wind_kts = random.randint(4, 38)
    visibility_km = round(random.uniform(0.6, 15.0), 1)
    is_severe = condition in ("HEAVY_SNOW", "THUNDERSTORM", "FOG") or wind_kts >= 30

    weather_records.append({
        "observation_id": str(uuid.uuid4()),
        "airport_iata": ap["airport_iata"],
        "observed_at_utc": observed_at,
        "temperature_c": round(random.uniform(-8.0, 32.0), 1),
        "wind_speed_kts": wind_kts,
        "visibility_km": visibility_km,
        "weather_condition": condition,
        "is_severe_weather": is_severe,
    })

weather_file = f"{WEATHER_DIR}/weather_{batch_ts}.json"
with open(weather_file, mode="w", encoding="utf-8") as f:
    for rec in weather_records:
        f.write(json.dumps(rec) + "\n")

print(f"Wrote {len(weather_records)} weather records to {weather_file}")

In [0]:
# ---------------------------------------------------------------------------
# 4. Generate Synthetic Cargo Manifests Linked to Active Flights (JSON Lines)
# ---------------------------------------------------------------------------
# Pull real callsigns from Bronze telemetry so future Silver/Gold joins match
observed_callsigns = [
    row["callsign"].strip()
    for row in (
        spark.table(BRONZE_TELEMETRY_TABLE)
        .select(F.trim(F.get_json_object(F.col("payload_json"), "$.callsign")).alias("callsign"))
        .filter(F.col("callsign").isNotNull() & (F.col("callsign") != ""))
        .distinct()
        .limit(15)
        .collect()
    )
]

fallback_callsigns = ["LH8301", "LH8210", "KL0887", "FX5012", "5X0214"]
active_callsigns = list(dict.fromkeys(observed_callsigns + fallback_callsigns))

cargo_categories = ["PHARMA_COLD_CHAIN", "ELECTRONICS", "AUTO_PARTS", "PERISHABLES", "GENERAL_CARGO"]
priorities = ["CRITICAL", "EXPRESS", "STANDARD"]
iata_codes = [ap["airport_iata"] for ap in airports_data]

manifest_records = []
created_at = datetime.now(timezone.utc).isoformat()

for idx, callsign in enumerate(active_callsigns, start=1):
    origin, destination = random.sample(iata_codes, 2)
    category = random.choice(cargo_categories)
    manifest_records.append({
        "manifest_id": f"AWB-{batch_ts[:8]}-{idx:04d}",
        "flight_callsign": callsign,
        "origin_iata": origin,
        "destination_iata": destination,
        "cargo_category": category,
        "priority_level": "CRITICAL" if category == "PHARMA_COLD_CHAIN" else random.choice(priorities),
        "total_weight_kg": round(random.uniform(1200.0, 48500.0), 1),
        "declared_value_usd": round(random.uniform(25000.0, 1500000.0), 2),
        "temp_controlled_required": category in ("PHARMA_COLD_CHAIN", "PERISHABLES"),
        # Synthetic PII fields intentionally added for Phase 3 Unity Catalog masking
        "shipper_email": f"dispatch.ops{idx}@aircargo-partner.example.com",
        "consignee_phone": f"+48-500-{random.randint(100, 999)}-{random.randint(100, 999)}",
        "manifest_created_utc": created_at,
    })

manifests_file = f"{MANIFESTS_DIR}/cargo_manifests_{batch_ts}.json"
with open(manifests_file, mode="w", encoding="utf-8") as f:
    for rec in manifest_records:
        f.write(json.dumps(rec) + "\n")

print(f"Wrote {len(manifest_records)} cargo manifests to {manifests_file}")

In [0]:
# ---------------------------------------------------------------------------
# 5. Verify Generated Files in Unity Catalog Volume
# ---------------------------------------------------------------------------
for folder in [AIRPORTS_DIR, WEATHER_DIR, MANIFESTS_DIR]:
    print(f"\n--- Contents of {folder} ---")
    for file_info in dbutils.fs.ls(folder):
        print(f"  {file_info.name} ({file_info.size} bytes)")